In [0]:
class Bronze_drivers():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import IntegerType, StringType, DateType, StructType, StructField
        name_schema= StructType([
                StructField("forename", StringType(), False),        # not  Nullable
                StructField("surname", StringType(), False)           # not Nullable
                ])
    
        schema = StructType([
                StructField("driverId", IntegerType(), False),        # Primary Key, NOT NULL
                StructField("driverRef", StringType(), False),        # Unique identifier, NOT NULL
                StructField("number", IntegerType(), True),           # Nullable
                StructField("code", StringType(), True),              # Nullable
                StructField("name", name_schema, False),              # NOT NULL       
                StructField("dob", DateType(), True),                 # Nullable
                StructField("nationality", StringType(), True),       # Nullable
                StructField("url", StringType(), False)               # NOT NULL
                ])

        return schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .load(f"{self.main_path}/formula1_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze dirvers Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('DriversIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-driver")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append") #full load so we need to use overwrite or complete mode n=but it is not supported by community edition
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('formula1_race.bronze.drivers')
                    ) 
        print("Done")
        return sQuery   


In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class MyListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        print(f"Batch ID: {event.progress['batchId']}")
        print(f"Rows Read: {event.progress['numInputRows']}")
        print(f"Duration (ms): {event.progress['durationMs']}")
        print(f"Rows/sec: {event.progress['processedRowsPerSecond']}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(MyListener())

In [0]:
source='Ergast API'
Bronze_drivers_instance = Bronze_drivers("drivers",source)
Squery_Bronze_drivers =Bronze_drivers_instance.process()
Squery_Bronze_drivers.awaitTermination()
print("Successfully bronze-ingestion-drivers stream in running")
Squery_Bronze_drivers.stop()